Run before everything

In [3]:
from ugot import ugot  # UGOT robot SDK
import time

# Create a robot instance and connect
got = ugot.UGOT()
got.initialize("192.168.88.1")  # Replace with your actual robot IP

# IMPORTANT: You must include "face_recognition" for Part 3 to work!
# We keep word and line recognition in case you are running from the fork.
got.load_models(
    ["word_recognition", "line_recognition", "face_recognition"]
)

# Open camera for face detection
got.open_camera()

# Optional: Set the face recognition name you are looking for
TARGET_NAME = "Target" 

print("Part 3 Models Loaded: Ready for Face Search & Approach")

192.168.88.1:50051
Part 3 Models Loaded: Ready for Face Search & Approach


Part3

In [5]:
import time
import cv2
import numpy as np
from ugot import ugot

# --- INITIALIZATION ---
got = ugot.UGOT()
got.initialize("192.168.88.1")
# Load models for the robot's own camera
got.load_models(["face_recognition"])
got.open_camera()

# --- Configuration Constants ---
TARGET_NAME = "Target" 
BASE_SPEED = 12         
APPROACH_GAP = 15       

def release_token_sequence():
    """Drops the token using tested angles from your notebooks."""
    print("Initiating Drop Sequence...")
    # Lower arm
    got.mechanical_joint_control(angle1=0, angle2=-30, angle3=-30, duration=800)
    time.sleep(1.5)
    # Release
    got.mechanical_clamp_release()
    print("Token Delivered.")
    time.sleep(1.0)
    # Reset
    got.mechanical_joint_control(angle1=0, angle2=45, angle3=45, duration=500)
    time.sleep(0.5)

def face_find_and_approach():
    """Robot uses its own camera to find the person and deliver."""
    print(f"Robot scanning for {TARGET_NAME}...")
    while True:
        res = got.get_face_recognition_total_info()
        if res is not None and len(res) > 1:
            detected_name = str(res[1]).upper()
            if TARGET_NAME.upper() in detected_name:
                got.mecanum_move_xyz(0, int(BASE_SPEED), 0)
                distance = res[6]
                if 0 < distance <= APPROACH_GAP:
                    got.mecanum_stop()
                    break
        else:
            got.mecanum_move_xyz(0, 0, 15) # Spin to search
        time.sleep(0.05)

def gesture_control_mode():
    """Opens your Mac camera to wait for a start signal/gesture."""
    print("Opening Mac Camera for Gesture Control...")
    cap = cv2.VideoCapture(0) # 0 is usually the built-in Mac camera
    
    if not cap.isOpened():
        print("Error: Could not open Mac Camera.")
        return False

    print("WAITING FOR GESTURE TO START MISSION... (Press 'q' to bypass)")
    
    start_mission = False
    while True:
        ret, frame = cap.read()
        if not ret: break
        
        # Simple placeholder for your pose/gesture logic:
        # In a real run, your 'pose_yolo' would analyze 'frame' here.
        cv2.imshow('Mac Camera - Gesture Control', frame)
        
        # Logic: If you press 's' or a gesture is detected, start Part 3
        key = cv2.waitKey(1) & 0xFF
        if key == ord('s'): 
            print("Gesture Received! Starting Mission...")
            start_mission = True
            break
        elif key == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    return start_mission

def part3run():
    # 1. Wait for User Gesture on Mac Camera
    if not gesture_control_mode():
        print("Mission aborted by user.")
        return

    print("Phase 3: Final Approach and Delivery Started")
    
    # 2. Search for person using Robot Camera
    face_find_and_approach()
    got.mecanum_stop()
    time.sleep(0.5)
    
    # 3. Final positioning nudge
    got.mecanum_move_xyz(0, 10, 0)
    time.sleep(0.7)
    got.mecanum_stop()
    
    # 4. Drop Token
    release_token_sequence()
    
    # 5. Finish
    got.screen_print("FINISHED")
    got.mecanum_move_xyz(0, -15, 0) 
    time.sleep(1.5)
    got.mecanum_stop()

if __name__ == "__main__":
    try:
        part3run()
    except Exception as e:
        print(f"System Error: {e}")
        got.mecanum_stop()

ImportError: dlopen(/Users/ireshramasamy/Desktop/National-Youth-Tech-Championship-2026/venv/lib/python3.13/site-packages/cv2/cv2.abi3.so, 0x0002): Library not loaded: @loader_path/libaom.3.12.1.dylib
  Referenced from: <F05BCF96-D65F-35CD-B375-6DB36BE761A1> /Users/ireshramasamy/Desktop/National-Youth-Tech-Championship-2026/venv/lib/python3.13/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib
  Reason: tried: '/Users/ireshramasamy/Desktop/National-Youth-Tech-Championship-2026/venv/lib/python3.13/site-packages/cv2/.dylibs/libaom.3.12.1.dylib' (no such file), '/usr/local/lib/libaom.3.12.1.dylib' (no such file), '/usr/lib/libaom.3.12.1.dylib' (no such file, not in dyld cache)